# Mr. Chicken + LatentSync (ByteDance) no Kaggle

Este notebook executa o **LatentSync** (sincronização labial baseada em difusão de última geração da ByteDance) como um microserviço REST, compatível com o Mr. Chicken local.


In [ ]:
from pathlib import Path
import os, secrets, subprocess, sys, time

WORK_DIR = Path('/kaggle/working')
REPO_DIR = WORK_DIR / 'LatentSync'
SERVICE_FILE = WORK_DIR / 'mrchicken_lipsync_service.py'
OUTPUTS_DIR = WORK_DIR / 'mrchicken_lipsync_outputs'
PORT = int(os.environ.get('LIPSYNC_PORT', '8010'))

PYTHON_EXE = '/opt/conda/bin/python' if os.path.exists('/opt/conda/bin/python') else sys.executable
os.environ['PYTHON'] = PYTHON_EXE
API_KEY = ""
os.environ['LIPSYNC_API_KEY'] = API_KEY
os.environ['MUSETALK_OUTPUTS_DIR'] = str(OUTPUTS_DIR)
os.environ['MUSETALK_TIMEOUT_SECONDS'] = '1800'
os.environ['HF_HOME'] = str(WORK_DIR / 'hf-cache')

WORK_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
print('WORK_DIR=', WORK_DIR)
print('REPO_DIR=', REPO_DIR)
print('OUTPUTS_DIR=', OUTPUTS_DIR)
print('PORT=', PORT)
print('PYTHON_EXE=', PYTHON_EXE)


In [ ]:
# Clone/update do LatentSync e dependências no ambiente do Kaggle (Python 3.10+)
import shutil, subprocess, sys, os
from pathlib import Path

giturl = 'https://github.com/bytedance/LatentSync.git'

if REPO_DIR.exists() and not (REPO_DIR / '.git').exists():
    shutil.rmtree(REPO_DIR)

if not REPO_DIR.exists():
    print("Clonando repositório LatentSync...")
    subprocess.run(['git', 'clone', giturl, str(REPO_DIR)], check=True)

# Tenta instalar pacotes de sistema necessários (libgl1)
try:
    print("Verificando/instalando libgl1 via apt...")
    subprocess.run(['sudo', 'apt-get', 'update', '-y'], check=False)
    subprocess.run(['sudo', 'apt-get', 'install', '-y', 'libgl1'], check=False)
except Exception as e:
    print("Aviso ao tentar atualizar apt:", e)

# Instala dependências do Python
print("Instalando pip dependencies...")
PYTHON_EXE = os.environ.get('PYTHON') or sys.executable
subprocess.run([PYTHON_EXE, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'], check=True)

packages = [
    'diffusers>=0.32.2',
    'transformers>=4.48.0',
    'decord>=0.6.0',
    'accelerate>=0.26.1',
    'einops>=0.7.0',
    'omegaconf>=2.3.0',
    'opencv-python>=4.9.0.80',
    'mediapipe>=0.10.11',
    'python_speech_features>=0.6',
    'librosa>=0.10.1',
    'scenedetect>=0.6.1',
    'ffmpeg-python>=0.2.0',
    'imageio>=2.31.1',
    'imageio-ffmpeg',
    'fastapi',
    'uvicorn',
    'python-multipart',
    'insightface',
    'onnxruntime-gpu',
    'DeepCache'
]

subprocess.run([PYTHON_EXE, '-m', 'pip', 'install', '-q'] + packages, check=True)

# Patch de FP16 para permitir rodar em float16 na GPU T4 (compute capability 7.5)
inference_file = REPO_DIR / 'scripts' / 'inference.py'
if inference_file.exists():
    content = inference_file.read_text(encoding='utf-8')
    content = content.replace('get_device_capability()[0] > 7', 'get_device_capability()[0] >= 7')
    inference_file.write_text(content, encoding='utf-8')
    print('Patch de FP16 aplicado com sucesso no scripts/inference.py!')
else:
    print('Aviso: scripts/inference.py nao encontrado para patch de FP16.')

print("Dependências do Python instaladas com sucesso!")


In [ ]:
# Download de checkpoints usando a biblioteca Python huggingface_hub
from pathlib import Path
import sys, subprocess, os

PYTHON_EXE = os.environ.get('PYTHON') or sys.executable
print("Garantindo instalação/atualização do huggingface_hub...")
subprocess.run([PYTHON_EXE, '-m', 'pip', 'install', '-q', '--upgrade', 'huggingface_hub'], check=True)

checkpoints_dir = REPO_DIR / 'checkpoints'
checkpoints_dir.mkdir(parents=True, exist_ok=True)

print("Baixando checkpoints de ByteDance/LatentSync programaticamente...")
from huggingface_hub import snapshot_download
try:
    snapshot_download(
        repo_id='ByteDance/LatentSync',
        local_dir=str(checkpoints_dir),
        ignore_patterns=['*.git*', 'README.md']
    )
    print("Checkpoints baixados com sucesso!")
except Exception as e:
    print("Erro ao baixar checkpoints via snapshot_download:", e)
    raise e


In [ ]:
SERVICE_SOURCE = r'''
import os
import re
import shutil
import subprocess
import time
from pathlib import Path
from typing import Optional
from fastapi import FastAPI, File, Form, Header, HTTPException, Request, UploadFile
from fastapi.responses import FileResponse
from pydantic import BaseModel, ConfigDict, Field

app = FastAPI(title="MrChicken LatentSync Service", version="1.0.0")

WORK_DIR = Path('/kaggle/working')
REPO_DIR = WORK_DIR / 'LatentSync'
OUTPUTS_DIR = WORK_DIR / 'mrchicken_lipsync_outputs'
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

class GenerateResponse(BaseModel):
    model_config = ConfigDict(populate_by_name=True)
    success: bool
    video_path: str = Field(..., alias='videoPath')
    video_url: Optional[str] = Field(default=None, alias='videoUrl')

def verify_api_key(authorization: Optional[str] = Header(default=None), x_api_key: Optional[str] = Header(default=None)) -> None:
    return  # Keyless

def safe_path_segment(value: str) -> str:
    return re.sub(r'[^a-zA-Z0-9._-]', '-', value)[:120] or 'job'

def safe_upload_name(filename: str | None, fallback: str) -> str:
    suffix = Path(filename or fallback).suffix.lower()
    if suffix not in {'.jpg', '.jpeg', '.png', '.webp', '.mp4', '.mov', '.webm', '.wav', '.mp3', '.m4a'}:
        suffix = Path(fallback).suffix
    return f'{Path(fallback).stem}{suffix}'

def save_upload(upload: UploadFile, destination: Path) -> Path:
    destination.parent.mkdir(parents=True, exist_ok=True)
    with destination.open('wb') as output:
        shutil.copyfileobj(upload.file, output)
    return destination
def run_command_streaming(cmd, cwd, env) -> tuple[int, str]:
    process = subprocess.Popen(
        cmd,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    output_lines = []
    for line in process.stdout:
        print(line, end='', flush=True)
        output_lines.append(line)
    process.wait()
    return process.returncode, ''.join(output_lines)

def run_latentsync(job_id: str, avatar_path: Path, audio_path: Path) -> Path:
    job_dir = OUTPUTS_DIR / safe_path_segment(job_id)
    job_dir.mkdir(parents=True, exist_ok=True)
    output_path = job_dir / "latentsync-output.mp4"
    
    avatar_ext = avatar_path.suffix.lower()
    is_image = avatar_ext in {'.jpg', '.jpeg', '.png', '.webp'}
    temp_video_path = None
    
    if is_image:
        print(f"Detectada imagem estática: {avatar_path}. Convertendo para vídeo estático com FFmpeg...")
        temp_video_path = job_dir / "temp_image_looped.mp4"
        
        cmd_ffmpeg = [
            'ffmpeg', '-y',
            '-loop', '1',
            '-i', str(avatar_path),
            '-i', str(audio_path),
            '-c:v', 'libx264',
            '-tune', 'stillimage',
            '-c:a', 'aac',
            '-pix_fmt', 'yuv420p',
            '-shortest',
            str(temp_video_path)
        ]
        
        print(f"Executando FFmpeg: {' '.join(cmd_ffmpeg)}")
        result_ffmpeg = subprocess.run(cmd_ffmpeg, capture_output=True, text=True, check=False)
        if result_ffmpeg.returncode != 0:
            print("Stderr FFmpeg:", result_ffmpeg.stderr)
            raise RuntimeError(f"FFmpeg falhou ao criar vídeo estático da imagem. Código {result_ffmpeg.returncode}")
        
        input_video_path = temp_video_path
        print(f"Vídeo estático gerado em: {input_video_path}")
    else:
        input_video_path = avatar_path
        
    config_candidate1 = REPO_DIR / "configs" / "unet" / "second_stage.yaml"
    config_candidate2 = REPO_DIR / "configs" / "unet" / "stage2.yaml"
    
    if config_candidate1.exists():
        config_path = "configs/unet/second_stage.yaml"
    elif config_candidate2.exists():
        config_path = "configs/unet/stage2.yaml"
    else:
        config_path = "configs/unet/stage2.yaml"
        
    cmd = [
        os.environ.get('PYTHON') or 'python',
        '-m', 'scripts.inference',
        '--unet_config_path', config_path,
        '--inference_ckpt_path', 'checkpoints/latentsync_unet.pt',
        '--video_path', str(input_video_path),
        '--audio_path', str(audio_path),
        '--video_out_path', str(output_path),
        '--inference_steps', '15',
        '--guidance_scale', '1.5'
    ]
    
    print(f"Executando LatentSync: {' '.join(cmd)}")
    env = os.environ.copy()
    env['PYTHONPATH'] = f"{REPO_DIR}:{env.get('PYTHONPATH', '')}"
    
    returncode, output = run_command_streaming(cmd, str(REPO_DIR), env)
    if returncode != 0:
        error_msg = f"LatentSync falhou com código {returncode}."
        if output:
            error_msg += f"\nLogs:\n{output[-2000:]}"
        raise RuntimeError(error_msg)
        
    if not output_path.exists():
        raise FileNotFoundError(f"Vídeo de saída do LatentSync não encontrado em {output_path}")
        
    if is_image and temp_video_path and temp_video_path.exists():
        try:
            temp_video_path.unlink()
        except Exception:
            pass
            
    return output_path

@app.get('/health')
def health() -> dict[str, object]:
    import torch
    return {'success': True, 'engine': 'latentsync', 'gpuAvailable': torch.cuda.is_available()}

@app.post('/generate-upload')
async def generate_upload(request: Request, jobId: str = Form(..., min_length=1), avatar: UploadFile = File(...), audio: UploadFile = File(...)):
    safe_job_id = safe_path_segment(jobId)
    job_dir = OUTPUTS_DIR / safe_job_id
    avatar_path = save_upload(avatar, job_dir / safe_upload_name(avatar.filename, 'avatar.jpg'))
    audio_path = save_upload(audio, job_dir / safe_upload_name(audio.filename, 'audio.wav'))
    import asyncio
    import json
    from fastapi.responses import StreamingResponse
    async def event_generator():
        loop = asyncio.get_running_loop()
        task = loop.run_in_executor(None, run_latentsync, safe_job_id, avatar_path, audio_path)
        while not task.done():
            yield ' '
            await asyncio.sleep(5)
        try:
            video_path = await task
            result = {
                'success': True,
                'videoPath': str(video_path),
                'videoUrl': str(request.url_for('download_output', job_id=safe_job_id, filename=video_path.name))
            }
        except Exception as exc:
            import traceback
            error_text = traceback.format_exc()
            (job_dir / 'error.log').write_text(error_text, encoding='utf-8')
            result = {'success': False, 'error': str(exc), 'code': 'LATENTSYNC_ERROR'}
        yield json.dumps(result)
    return StreamingResponse(event_generator(), media_type='application/json')

@app.get('/outputs/{job_id}/{filename}', name='download_output')
def download_output(job_id: str, filename: str) -> FileResponse:
    output_path = OUTPUTS_DIR / safe_path_segment(job_id) / Path(filename).name
    if not output_path.exists():
        raise HTTPException(status_code=404, detail={'success': False, 'error': f'Output não encontrado: {filename}', 'code': 'FILE_NOT_FOUND'})
    return FileResponse(output_path, media_type='video/mp4', filename=Path(filename).name)
'''
SERVICE_FILE.write_text(SERVICE_SOURCE, encoding='utf-8')
print('Serviço escrito em', SERVICE_FILE)


In [ ]:
# Inicia FastAPI localmente e expõe com Cloudflare Tunnel temporário.
import os, queue, re, stat, threading, subprocess, time, requests, sys
PYTHON_EXE = os.environ.get('PYTHON') or sys.executable

cloudflared = WORK_DIR / 'cloudflared'
if not cloudflared.exists():
    subprocess.run(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', '-O', str(cloudflared)], check=True)
    cloudflared.chmod(cloudflared.stat().st_mode | stat.S_IEXEC)

for name in ['server_proc', 'tunnel_proc']:
    proc = globals().get(name)
    if proc and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=5)
        except Exception:
            proc.kill()

server_proc = subprocess.Popen([PYTHON_EXE, '-m', 'uvicorn', 'mrchicken_lipsync_service:app', '--host', '0.0.0.0', '--port', str(PORT), '--proxy-headers'], cwd=str(WORK_DIR), env=os.environ.copy(), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

def stream_logs(prefix, proc):
    for line in proc.stdout:
        print(prefix, line, end='')
threading.Thread(target=stream_logs, args=('[uvicorn]', server_proc), daemon=True).start()
time.sleep(5)
resp = requests.get(f'http://127.0.0.1:{PORT}/health', timeout=15)
print('Health local:', resp.status_code, resp.text[:500])

tunnel_proc = subprocess.Popen([str(cloudflared), 'tunnel', '--url', f'http://127.0.0.1:{PORT}', '--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
q = queue.Queue()
def capture_tunnel():
    for line in tunnel_proc.stdout:
        print('[cloudflared]', line, end='')
        q.put(line)
threading.Thread(target=capture_tunnel, daemon=True).start()

PUBLIC_URL = None
pattern = re.compile(r'https://[-a-zA-Z0-9.]+\.trycloudflare\.com')
deadline = time.time() + 90
while time.time() < deadline and PUBLIC_URL is None:
    try:
        line = q.get(timeout=2)
    except queue.Empty:
        continue
    match = pattern.search(line)
    if match:
        PUBLIC_URL = match.group(0)
if not PUBLIC_URL:
    raise RuntimeError('Não consegui obter URL pública do cloudflared.')

print('\n=== CONFIGURE NO .env.local DO MRCHICKEN ===')
print(f'LIPSYNC_API_URL={PUBLIC_URL}')
print('# LIPSYNC_API_KEY não é necessária')
print('LIPSYNC_TRANSFER_MODE=upload')
print('LIPSYNC_TIMEOUT_MS=1800000')
print('Endpoint público:', PUBLIC_URL)
